# Treino do modelo de EPI

Treina um detector YOLO de **capacete** e **colete** com o dataset do Roboflow,
para substituir a analise de cor do EPI Detect (que confunde bone com capacete
e camiseta laranja com colete).

## Antes de rodar

1. **Ambiente de execucao → Alterar o tipo de ambiente → GPU T4.** Sem GPU o treino leva horas.
2. **Secrets (icone de chave na barra lateral) → Adicionar novo secret:**
   - Nome: `ROBOFLOW_API_KEY`
   - Valor: a sua API key do Roboflow
   - Ative o acesso do notebook ao secret.

   Assim a chave nao fica escrita no notebook.
3. Preencha a celula **Configuracao** com o workspace, o projeto e a versao do Roboflow.
4. **Ambiente de execucao → Executar tudo.**

No fim, o arquivo `epi_v<VERSAO>.pt` e baixado. Renomeie para `models/epi.pt` no projeto.
Cada versao treina numa pasta propria (`epi_v1`, `epi_v2`...), entao os resultados anteriores nao sao apagados.

In [ ]:
# Confere se a GPU esta ativa. Se der erro, volte ao passo 1 acima.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Mesma versao do ultralytics usada no projeto, para os pesos abrirem sem surpresa
!pip install -q "ultralytics==8.4.173" roboflow

In [ ]:
#@title Configuracao
# Aparece na URL do Roboflow: app.roboflow.com/<WORKSPACE>/<PROJETO>/<VERSAO>
WORKSPACE = "seu-workspace"  #@param {type:"string"}
PROJETO = "construction-site-safety-xxxxx"  #@param {type:"string"}
VERSAO = 1  #@param {type:"integer"}

# yolo11n e o menor e mais rapido; roda bem em CPU depois de treinado.
MODELO_BASE = "yolo11n.pt"  #@param ["yolo11n.pt", "yolo11s.pt", "yolov8n.pt"]
EPOCAS = 150  #@param {type:"integer"}
IMGSZ = 640  #@param {type:"integer"}

# Salvar no Google Drive evita perder o treino se o Colab desconectar.
SALVAR_NO_DRIVE = True  #@param {type:"boolean"}

In [ ]:
import os

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/epi-detect-treino"
else:
    PASTA = "/content/epi-detect-treino"
os.makedirs(PASTA, exist_ok=True)
print("Resultados em:", PASTA)

In [ ]:
import yaml
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJETO).version(VERSAO).download("yolov8")

# O data.yaml exportado pelo Roboflow usa caminhos relativos que quebram no
# Colab; reescreve com caminhos a partir da pasta do dataset.
DATA = os.path.join(dataset.location, "data.yaml")
with open(DATA) as f:
    data = yaml.safe_load(f)
data["path"] = dataset.location
data["train"] = "train/images"
data["val"] = "valid/images"
TEM_TESTE = os.path.isdir(os.path.join(dataset.location, "test", "images"))
if TEM_TESTE:
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(DATA, "w") as f:
    yaml.safe_dump(data, f)

print("Classes:", data["names"])
for split in ("train", "valid", "test"):
    pasta = os.path.join(dataset.location, split, "images")
    if os.path.isdir(pasta):
        print(f"{split:6s}: {len(os.listdir(pasta))} imagens")

## Treino

Leva cerca de 1 a 2 horas numa T4, dependendo do tamanho do dataset.
O `patience` interrompe sozinho se o modelo parar de melhorar.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODELO_BASE)
model.train(
    data=DATA,
    epochs=EPOCAS,
    imgsz=IMGSZ,
    batch=16,
    patience=20,
    project=PASTA,
    name=f"epi_v{VERSAO}",
    exist_ok=True,
)
BEST = os.path.join(PASTA, f"epi_v{VERSAO}", "weights", "best.pt")
print("Melhor modelo:", BEST)

## Avaliacao

- **mAP50**: acerto geral (0 a 1). Acima de 0.7 ja e um bom comeco.
- Olhe por classe: `NO-Hardhat` costuma ser a mais dificil, e e a que importa para o caso do bone.
- Na **matriz de confusao**, a diagonal sao os acertos; fora dela, o que o modelo confunde com o que.

In [ ]:
best = YOLO(BEST)
metricas = best.val(data=DATA, split="test" if TEM_TESTE else "val")
print(f"mAP50    : {metricas.box.map50:.3f}")
print(f"mAP50-95 : {metricas.box.map:.3f}")
print()
for i, nome in best.names.items():
    if i < len(metricas.box.ap50):
        print(f"  {nome:16s} AP50 = {metricas.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display

pasta_treino = os.path.join(PASTA, f"epi_v{VERSAO}")
for arquivo in ("results.png", "confusion_matrix_normalized.png"):
    caminho = os.path.join(pasta_treino, arquivo)
    if os.path.exists(caminho):
        print(arquivo)
        display(Image(caminho, width=900))

In [ ]:
# Algumas previsoes em imagens que o modelo nao viu no treino
import glob

import cv2

split = "test" if TEM_TESTE else "valid"
amostras = sorted(glob.glob(os.path.join(dataset.location, split, "images", "*")))[:6]
for r in best.predict(amostras, conf=0.35, verbose=False):
    display(Image(data=cv2.imencode(".jpg", r.plot())[1].tobytes(), width=600))

## Baixar o modelo

O arquivo baixado se chama `epi_v<VERSAO>.pt`. Copie para a pasta `models/` do projeto
e renomeie para `models/epi.pt` (guarde o anterior com outro nome, para comparar). Depois rode `python detect.py`: no terminal deve aparecer
`Metodo de EPI: modelo`.

In [ ]:
import shutil

from google.colab import files

shutil.copy(BEST, f"/content/epi_v{VERSAO}.pt")
files.download(f"/content/epi_v{VERSAO}.pt")